In [ ]:
# Una consulta conserva el origen para que la respuesta pueda verificarse y repetirse.

from pathlib import Path
import json
import sqlite3

import pandas as pd

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
DATABASE_FILE = ACTIVITY_DIR / "data" / "scopus_proptech.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# El esquema delimita qué puede consultarse antes de escribir la primera pregunta.

database = sqlite3.connect(DATABASE_FILE)
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", database)

In [ ]:
# La primera consulta mantiene una fila por documento y hace visible el orden de lectura.

documents = pd.read_sql_query(
    """
    SELECT title, publication_year, document_type
    FROM documents
    ORDER BY publication_year DESC, title
    LIMIT 10
    """,
    database,
)
documents

In [ ]:
# Filtrar el periodo responde una pregunta concreta sin transformar la fuente.

documents_recent = pd.read_sql_query(
    """
    SELECT title, publication_year, document_type
    FROM documents
    WHERE publication_year >= 2020
    ORDER BY publication_year DESC, title
    """,
    database,
)
documents_recent.head()

In [ ]:
# Agrupar permite comparar categorías sin perder la definición de cada grupo.

documents_by_type = pd.read_sql_query(
    """
    SELECT document_type, COUNT(*) AS document_count
    FROM documents
    GROUP BY document_type
    ORDER BY document_count DESC, document_type
    """,
    database,
)
documents_by_type

In [ ]:
# Los archivos persistentes convierten las respuestas de la sesión en entregables auditables.

documents_recent.to_csv(SUBMISSION_DIR / "documents_recent.csv", index=False)
documents_by_type.to_csv(SUBMISSION_DIR / "documents_by_type.csv", index=False)

questions = [
    {"pregunta": "¿Qué documentos de proptech se publicaron desde 2020?", "archivo_respuesta": "documents_recent.csv"},
    {"pregunta": "¿Qué tipos de documento componen el resultado de la búsqueda?", "archivo_respuesta": "documents_by_type.csv"},
]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
database.close()